In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics opencv-python filterpy supervision

In [ ]:
%cd /content/drive/MyDrive/Avzdax

/content/drive/MyDrive/Avzdax


In [ ]:


import argparse
import time
import math
from collections import defaultdict, deque

import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
import itertools
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

# --------------------------------------------------------------
# Speed estimator per track
# --------------------------------------------------------------
class SpeedEstimator:
    def __init__(self, max_history=30):
        self.history = defaultdict(lambda: deque(maxlen=max_history))

    def update(self, track_id, bbox, frame_idx):
        x1, y1, x2, y2 = bbox
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        self.history[track_id].append((frame_idx, cx, cy))

    def get_speed(self, track_id, fps, window=10):
        h = self.history[track_id]
        if len(h) < 2:
            return 0.0
        vals = list(h)[-window:]
        if len(vals) < 2:
            return 0.0
        t0, x0, y0 = vals[0]
        t1, x1, y1 = vals[-1]
        dt = (t1 - t0) / float(fps)
        if dt <= 0:
            return 0.0
        dist = math.hypot(x1 - x0, y1 - y0)
        return dist / dt


def closest_triplet_centroid(points, eps=50):
    """
    Fallback method using spatial indexing
    """
    points = np.array(points)

    # Use spatial indexing for efficient nearest neighbor search
    nn = NearestNeighbors(n_neighbors=min(8, len(points)), metric='euclidean')
    nn.fit(points)

    best_dist = float("inf")
    best_triplet = None

    for i, point in enumerate(points):
        distances, indices = nn.kneighbors([point])

        # Get closest neighbors (excluding the point itself)
        neighbor_indices = indices[0][1:]

        # Check combinations of the closest 5 neighbors
        for j in range(min(5, len(neighbor_indices))):
            for k in range(j + 1, min(6, len(neighbor_indices))):
                idx1, idx2 = neighbor_indices[j], neighbor_indices[k]
                triplet = np.array([points[i], points[idx1], points[idx2]])

                # Calculate compactness
                d01 = np.linalg.norm(triplet[0] - triplet[1])
                d02 = np.linalg.norm(triplet[0] - triplet[2])
                d12 = np.linalg.norm(triplet[1] - triplet[2])
                total_dist = d01 + d02 + d12


                if total_dist < best_dist and all(d < eps * 2 for d in [d01, d02, d12]):
                    best_dist = total_dist
                    best_triplet = triplet

    return np.mean(best_triplet, axis=0) if best_triplet is not None else np.mean(points[:3], axis=0)

# --------------------------------------------------------------
# Main processing function
# --------------------------------------------------------------

def process_video(input_path, output_path, model_name='yolov8m.pt', device=None,
                  conf_thresh=0.25, iou_thresh=0.45,
                  speed_thresh_pixels_per_second=20.0,
                  motion_window=8):

    model = YOLO(model_name)
    if device is not None:
        model.overrides['device'] = device

    byte_tracker = sv.ByteTrack()
    speed_est = SpeedEstimator(max_history=motion_window*3)

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video {input_path}")

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    frame_idx = 0
    t0 = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # YOLO inference
        results = model.predict(frame, imgsz=640, conf=conf_thresh, iou=iou_thresh, verbose=False)
        res = results[0]

        detections = sv.Detections.from_ultralytics(res)

        # Filter to person
        mask = detections.class_id == 0
        detections = detections[mask]

        # Apply ByteTrack
        tracked = byte_tracker.update_with_detections(detections)

        # Draw + update speed and collect stationary centroids

        stationary_centroids = []

        for i in range(len(tracked)):
            tid = tracked.tracker_id[i]
            x1, y1, x2, y2 = map(int, tracked.xyxy[i])

            # update motion
            speed_est.update(tid, tracked.xyxy[i], frame_idx)
            speed = speed_est.get_speed(tid, fps, window=motion_window)

            label = "walking" if speed >= speed_thresh_pixels_per_second else "stationary"
            color = (0, 200, 0) if label == "walking" else (0, 120, 255)

            # store stationary positions
            if label == "stationary":
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                stationary_centroids.append((cx, cy))

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            txt = f"ID:{tid} {label} {int(speed)}px/s"
            cv2.putText(frame, txt, (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

        # --------------------------------------------------------------
        # Detect cluster among stationary
        # --------------------------------------------------------------
        if len(stationary_centroids) >= 4:
          centroid = closest_triplet_centroid(stationary_centroids)

          if centroid is not None:
            cx, cy = centroid
            cv2.circle(frame, (int(cx), int(cy)), 50, (0, 0, 255), 5)
            cv2.putText(frame, "CLUSTER DETECTED", (50, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (0, 0, 255), 2)
        out.write(frame)

        if frame_idx % 50 == 0:
            print(f"Processed {frame_idx}/{total_frames}")

    cap.release()
    out.release()
    print("Saved:", output_path)



In [ ]:
input_path = "/content/drive/MyDrive/Avzdax/6387-191695740_small.mp4"
output_path = "newalgorithm.mp4"


process_video(input_path, output_path, model_name='yolov8l.pt',
                  conf_thresh=0.20, iou_thresh=0.40,
                  speed_thresh_pixels_per_second=10.0,
                  motion_window=15)

Processed 50/341
Processed 100/341
Processed 150/341
Processed 200/341
Processed 250/341
Processed 300/341
Saved: newalgorithm.mp4
